<a href="https://colab.research.google.com/github/c-marq/AI-Thinking-CAI1001C/blob/main/10-Computer-Vision/Guided-Project/GP10_Computer_Vision_Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GP10: Computer Vision — Object Detection with YOLO

**CAI1001C: Artificial Intelligence (AI) Thinking**  
**Chapter 10: Computer Vision**  
**Guided Project — Reference Material (Not Graded)**

---

In this guided project, you will:

1. Load a pre-trained object detection model (YOLO11) that already knows 80 object categories
2. Feed images to the model and see it draw bounding boxes around every object it detects
3. Experiment with **confidence thresholds** to control how selective the model is
4. Upload your own images and test the model's strengths and limitations
5. Explore the 80 COCO categories and understand why the model can't detect everything

**Key concept:** Object detection doesn't just say "there's a dog in the photo" — it says "there's a dog HERE" by drawing a bounding box with a confidence score. The confidence threshold is the single most important parameter you control.

---

## Setup

Run this cell first — it installs the Ultralytics library (which includes YOLO) and imports everything we need. The model weights (~6MB) download automatically the first time you load the model.

In [ ]:
# ============================================
# Run this cell first — installs and imports
# ============================================

!pip install ultralytics -q

from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import urllib.request

print("✅ Setup complete!")

## Step 1: Load the Pre-Trained YOLO11 Model

YOLO11n is the "nano" variant — the smallest and fastest version. It's **pre-trained** on the COCO dataset, which means someone else already trained it on hundreds of thousands of labeled images covering 80 object categories. We're downloading their finished model and using it directly.

Think of it like hiring an experienced chef who already knows 80 recipes. We just hand them ingredients (images) and they cook (detect objects).

In [ ]:
# ============================================
# Step 1: Load the pre-trained YOLO11 model
# ============================================

# yolo11n.pt = "nano" version — smallest, fastest, pre-trained on 80 categories
model = YOLO("yolo11n.pt")

print(f"✅ Model loaded: yolo11n.pt")
print(f"   Number of categories: {len(model.names)}")
print(f"   First 10 categories: {list(model.names.values())[:10]}")

## Step 2: Your First Object Detection

Let's feed an image to YOLO and see what it finds. The model will:
1. Process the entire image in a single pass (that's what "You Only Look Once" means)
2. Draw **bounding boxes** around every object it detects
3. Label each box with a category name and **confidence score**

Watch for the confidence scores — they tell you how sure the model is about each detection.

In [ ]:
# ============================================
# Step 2: Run detection on a street scene image
# ============================================

# Run detection (image downloads automatically from URL)
results = model("https://ultralytics.com/images/bus.jpg", verbose=False)
result = results[0]

# Display the image with bounding boxes drawn
annotated = result.plot()
plt.figure(figsize=(10, 7))
plt.imshow(annotated[:, :, ::-1])  # Convert BGR to RGB for display
plt.title("Object Detection Results — Street Scene", fontsize=14, fontweight="bold")
plt.axis("off")
plt.show()

# Print what was detected
print(f"\n📊 Detection Results:")
print(f"   Objects found: {len(result.boxes)}")
print(f"{'─' * 55}")
print(f"   {'#':<4} {'Object':<15} {'Confidence':<15} {'Box (x1,y1,x2,y2)'}")
print(f"{'─' * 55}")

for i, box in enumerate(result.boxes):
    cls_name = result.names[int(box.cls[0])]
    conf = float(box.conf[0])
    coords = box.xyxy[0].tolist()
    print(f"   {i+1:<4} {cls_name:<15} {conf:<15.1%} [{coords[0]:.0f}, {coords[1]:.0f}, {coords[2]:.0f}, {coords[3]:.0f}]")

# Expected Output (approximate — exact values may vary slightly):
# Objects found: 5
#    1    bus             ~94%
#    2    person          ~89%
#    3    person          ~88%
#    4    person          ~86%
#    5    person          ~62%

### What Just Happened?

The model found **5 objects** — a bus and four people — each with:
- A **bounding box** showing exactly WHERE in the image the object is (the colored rectangle)
- A **label** identifying WHAT the object is (bus, person)
- A **confidence score** showing HOW SURE the model is (94% for the bus, ~62% for one person)

Notice the range: the bus at ~94% — very confident. The fourth person at ~62% — less confident. Maybe that person is partially hidden, far away, or in an unusual pose. The model does its best, but some detections are stronger than others.

> **Note:** Exact percentages may vary by a point or two each run due to floating-point differences across hardware. This is normal — focus on the pattern, not the exact numbers.

## Step 3: Try a Different Image

Let's run detection on a second image to see how results change with different content.

In [ ]:
# ============================================
# Step 3: Detection on a different image
# ============================================

results2 = model("https://ultralytics.com/images/zidane.jpg", verbose=False)
result2 = results2[0]

# Display
annotated2 = result2.plot()
plt.figure(figsize=(10, 7))
plt.imshow(annotated2[:, :, ::-1])
plt.title("Object Detection Results — People", fontsize=14, fontweight="bold")
plt.axis("off")
plt.show()

# Print detections
print(f"\n📊 Detection Results:")
print(f"   Objects found: {len(result2.boxes)}")
print(f"{'─' * 55}")
for i, box in enumerate(result2.boxes):
    cls_name = result2.names[int(box.cls[0])]
    conf = float(box.conf[0])
    print(f"   {i+1:<4} {cls_name:<15} {conf:.1%}")

# Expected Output (approximate):
# Objects found: 3
#    1    person          ~84%
#    2    person          ~78%
#    3    tie             ~45%

### Notice the Tie

The model detected a "tie" at about 45% confidence. That's a low score. Is it really a tie, or is the model guessing? This is a useful teaching moment:

- **High confidence (>80%):** The model is pretty sure. These are usually correct.
- **Medium confidence (50–80%):** Worth checking. Might be right, might be wrong.
- **Low confidence (<50%):** The model is uncertain. Could be correct, but don't bet on it.

What if we could filter out low-confidence detections? That's exactly what the **confidence threshold** does — and it's the most important concept in today's demo.

## Step 4: The Confidence Threshold Experiment

This is the key interactive moment. The **confidence threshold** tells the model: "Only show me detections where you're at least THIS confident."

- **Low threshold (25%):** Show everything, even uncertain guesses
- **Medium threshold (50%):** Show only moderately confident detections
- **High threshold (75%):** Show only very confident detections

Watch what happens to the number of detected objects as we raise the bar.

In [ ]:
# ============================================
# Step 4: Compare three confidence thresholds
# ============================================

thresholds = [0.25, 0.50, 0.75]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, conf_threshold in enumerate(thresholds):
    results = model("https://ultralytics.com/images/bus.jpg",
                     conf=conf_threshold, verbose=False)
    annotated = results[0].plot()

    axes[idx].imshow(annotated[:, :, ::-1])
    axes[idx].set_title(
        f"Threshold ≥ {conf_threshold:.0%}\n"
        f"({len(results[0].boxes)} objects detected)",
        fontsize=12, fontweight="bold"
    )
    axes[idx].axis("off")

plt.suptitle("Effect of Confidence Threshold on Detection",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n📊 Threshold Comparison:")
print("   25% threshold → 5 objects (catches everything)")
print("   50% threshold → 5 objects (still catches all)")
print("   75% threshold → 4 objects (the ~62% person drops out)")
print()
print("🔑 The person at ~62% confidence DISAPPEARS at 75%.")
print("   That person is real — but the model isn't confident enough.")
print("   What if this were a security camera? You just lost a detection.")

# Expected Output:
# Three side-by-side panels: 5 objects, 5 objects, 4 objects
# The least-confident person disappears at the 75% threshold

### The Precision-Coverage Tradeoff

Think of it like a smoke detector's sensitivity:
- **Too sensitive** → goes off every time you cook (annoying false alarms)
- **Not sensitive enough** → misses a real fire (dangerous)

There's no universally "right" threshold. It depends on what you're using the detector for:

| Application | Recommended Threshold | Why |
|---|---|---|
| Security camera (school) | **Low** (~25%) | Missing a person is worse than a false alarm |
| Medical scan flagging | **Low** (~20%) | Missing a tumor is worse than an extra test |
| Automated checkout | **High** (~75%) | Billing errors frustrate customers |
| Social media auto-tagging | **Medium** (~50%) | Balance between coverage and accuracy |

## ▶ Your Turn: Experiment with Thresholds

Modify the threshold value below and re-run the cell to see how it affects detection.

In [ ]:
# ============================================
# ▶ YOUR TURN: Change the threshold and re-run
# ============================================

# ▶ MODIFY THIS VALUE (try: 0.10, 0.30, 0.60, 0.80, 0.90)
my_threshold = 0.25

# Run detection with your threshold
results = model("https://ultralytics.com/images/bus.jpg",
                 conf=my_threshold, verbose=False)
result = results[0]

# Display
annotated = result.plot()
plt.figure(figsize=(10, 7))
plt.imshow(annotated[:, :, ::-1])
plt.title(f"Threshold = {my_threshold:.0%} — {len(result.boxes)} objects detected",
          fontsize=14, fontweight="bold")
plt.axis("off")
plt.show()

# Print what was found
print(f"\n📊 At {my_threshold:.0%} confidence threshold:")
for i, box in enumerate(result.boxes):
    cls_name = result.names[int(box.cls[0])]
    conf = float(box.conf[0])
    print(f"   {i+1}. {cls_name} ({conf:.1%})")

if len(result.boxes) == 0:
    print("   No detections! Your threshold may be too high.")
    print("   Try lowering it — even confident models rarely hit 95%+.")

## Step 5: Upload Your Own Image

Now let's see what YOLO makes of YOUR world. Upload a photo from your phone or computer — your desk, a street view, your lunch, anything. See what the model detects and what it misses.

In [ ]:
# ============================================
# Step 5: Upload and test your own image
# ============================================

from google.colab import files

print("📷 Upload an image to test the object detector!")
print("   Ideas: your desk, a kitchen, a parking lot, your lunch...\n")

uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    print(f"\n✅ Uploaded: {filename}")

    # Run detection
    results = model(filename, verbose=False)
    result = results[0]

    # Display results
    annotated = result.plot()
    plt.figure(figsize=(10, 7))
    plt.imshow(annotated[:, :, ::-1])
    plt.title(f"Your Image — {len(result.boxes)} Objects Detected",
              fontsize=14, fontweight="bold")
    plt.axis("off")
    plt.show()

    # Print what was found
    print(f"\n📊 Found {len(result.boxes)} objects:")
    for i, box in enumerate(result.boxes):
        cls_name = result.names[int(box.cls[0])]
        conf = float(box.conf[0])
        print(f"   {i+1}. {cls_name} ({conf:.1%} confidence)")

    if len(result.boxes) == 0:
        print("   No objects detected! Try a different image or lower the threshold.")
        print("   Hint: results = model(filename, conf=0.15, verbose=False)")

## ▶ Your Turn: Test Your Image at Different Thresholds

If you uploaded an image above, try running it at different confidence levels. Replace `"your_filename.jpg"` with the actual filename from your upload.

In [ ]:
# ============================================
# ▶ YOUR TURN: Test your uploaded image at different thresholds
# ============================================

# ▶ MODIFY: Replace with your uploaded filename
my_image = "WhatsApp Image 2026-02-10 at 10.46.20 PM.jpeg"  # ← change this!

# ▶ MODIFY: Try different thresholds
my_threshold = 0.25

results = model(my_image, conf=my_threshold, verbose=False)
result = results[0]

annotated = result.plot()
plt.figure(figsize=(10, 7))
plt.imshow(annotated[:, :, ::-1])
plt.title(f"Your Image — threshold {my_threshold:.0%} — {len(result.boxes)} objects",
          fontsize=14, fontweight="bold")
plt.axis("off")
plt.show()

for i, box in enumerate(result.boxes):
    cls_name = result.names[int(box.cls[0])]
    conf = float(box.conf[0])
    print(f"   {i+1}. {cls_name} ({conf:.1%})")

## Step 6: What Can YOLO Detect? (And What Can't It?)

YOLO11n is trained on the COCO dataset — exactly **80 object categories**. That's its entire menu. If an object isn't on this list, the model will either miss it or misclassify it as the closest category it knows.

For example, when we tested with a driver's license, the model detected:
- "book" (67.3% confidence) — the card shape looks vaguely like a book
- "person" (62.5% confidence) — the photo on the license

The model can never say "I don't know." It always picks the best match from its 80 options.

In [ ]:
# ============================================
# Step 6: View all 80 COCO categories
# ============================================

print("🏷️  All 80 COCO Categories YOLO Can Detect:\n")

for i, name in model.names.items():
    print(f"   {i:>2}: {name}", end="")
    if (i + 1) % 5 == 0:
        print()

print("\n\n🤔 What's NOT on this list?")
print("   ❌ No croquetas, tostón press, or plantains")
print("   ❌ No specific dog breeds (just 'dog')")
print("   ❌ No Miami Dolphins jersey (just 'person')")
print("   ❌ No driver's license, textbook, or homework")
print()
print("   → The model only detects what it was trained on!")
print("   → This is the same 'training data' principle from every chapter.")

## Summary: What We Learned

| Concept | What It Means |
|---|---|
| **Object Detection** | Finding and locating multiple objects in one image — not just labeling the whole image |
| **Bounding Box** | The rectangle showing WHERE the detected object is |
| **Confidence Score** | How sure the model is about each detection (0–100%) |
| **Confidence Threshold** | The cutoff — only show detections above this confidence level |
| **Pre-trained Model** | A model trained by someone else that we use directly — YOLO11n knows 80 COCO categories |
| **The Tradeoff** | Lower threshold = more detections but more errors. Higher threshold = fewer but more reliable detections. |

### The Big Idea

The confidence threshold is not a technical setting — it's a **design decision**. The right threshold depends on the real-world consequences of missing something vs. flagging something incorrectly. This is a human judgment, not a math problem.

---

**Next:** In the group lab, you'll use **Google AI Studio** to explore a completely different approach to computer vision — instead of bounding boxes and fixed categories, you'll have a conversation with a multimodal AI about what it sees in your images.